# Per-cell mechanistic fits — parameter-space exploration

Each of the 77 HD cells was fit independently with **all 32 mechanistic params** (its own attractor geometry + evoked response), warm-started from the population best (`_full_fit_result.json`) — mirroring the phenomenological DoVM per-cell workflow. (A shared frozen attractor was tried and rejected: the population ring is a ~40 Hz-regime network and goes degenerate/multi-lobe at the low drive that low-baseline cells need — cells span baseline 0–53 Hz with FC/baseline ratios 4×–35×, too heterogeneous for one ring.)

This notebook explores how the per-cell params distribute, how they relate to each cell's directional tuning, and decomposes what each model component (FC / dip / SC-NMDA / channels / adaptation) contributes to the response.

Pipeline: `extract_percell_targets.py` → `fit_cell_mech.py` (SLURM array) → `compile_mech_results.py` → this notebook.

> Per-cell R² is expected **below** the population 0.921 — single-cell PSTHs are much noisier than the population mean, and the biophysically-constrained model can fit some cells worse than a flexible phenomenological one. Where mechanistic R² falls short of DoVM (last cell) is itself a finding, not a regression.

In [ ]:
import json
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

df = pd.read_parquet('optimized_cells_mech.parquet')
npz = np.load('vivo_percell.npz', allow_pickle=True)
print(df.shape, '| valid:', int(df['valid'].sum()) if 'valid' in df else 'n/a')

GEOM = ['tau_E','tau_I','I_baseline','J1','KAPPA_E','W_IE','KAPPA_I','W_EI','I_HD','W_GLOBAL']
EVOKED = ['A_fast','fc_stim_duration','stim_delay','reversal_potential','g_T','V_half_T',
          'k_T','tau_hT','E_Ca','g_h','V_half_h','tau_h_on','tau_h_off','g_a','tau_a',
          'A_sc','tau_sc_on','tau_sc_off','sc_delay','tau_scg','tau_sc_off2','w_sc']
# spotlight: baseline drive (I_HD/I_baseline/J1) + SC-NMDA (A_sc/tau_sc*) + FC + adaptation
SPOTLIGHT = ['I_HD','J1','A_fast','A_sc','tau_sc_off','tau_sc_off2','g_a','g_T','g_h']
TUNING = [c for c in ['HDpeakFR','HDInx','HDAngle','HDFR','nSpk'] if c in df.columns]
df[['Animal_Id','Cell_Id','R_Squared']+SPOTLIGHT].head()

## Fit quality across cells

In [ ]:
fig, ax = plt.subplots(figsize=(6,3.5))
ax.hist(df['R_Squared'].clip(-1,1), bins=25, color='0.4')
ax.axvline(0.921, color='crimson', ls='--', label='population fit (0.921)')
ax.set_xlabel('per-cell R²'); ax.set_ylabel('# cells'); ax.legend(); ax.set_title('per-cell fit quality')
plt.tight_layout(); plt.show()
print(df['R_Squared'].describe()[['mean','50%','min','max']])

## Evoked-parameter distributions across the 77 cells
Spotlight the SC/NMDA (`A_sc`, `tau_sc_off`, `tau_sc_off2`, `w_sc`), FC (`A_fast`), adaptation (`g_a`, `tau_a`) and channel (`g_T`, `g_h`) knobs.

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(11,9))
for ax, p in zip(axes.ravel(), SPOTLIGHT):
    ax.hist(df[p], bins=20, color='steelblue')
    ax.axvline(df[p].median(), color='crimson', ls='--')
    ax.set_title(p, fontsize=10)
fig.suptitle('evoked-param distributions (red = median)')
plt.tight_layout(); plt.show()

## Does evoked structure track directional tuning?
e.g. is SC amplitude (`A_sc`) related to tuning sharpness (`HDInx`) or peak rate (`HDpeakFR`)?

In [ ]:
pairs = [('A_sc','HDpeakFR'), ('A_sc','HDInx'), ('A_fast','HDpeakFR'), ('g_a','HDInx')]
pairs = [(x,y) for x,y in pairs if y in df.columns]
fig, axes = plt.subplots(1, len(pairs), figsize=(4*len(pairs), 3.5))
axes = np.atleast_1d(axes)
for ax,(x,y) in zip(axes, pairs):
    ax.scatter(df[x], df[y], s=18, alpha=0.7)
    r = df[[x,y]].corr().iloc[0,1]
    ax.set_xlabel(x); ax.set_ylabel(y); ax.set_title(f'r = {r:.2f}')
plt.tight_layout(); plt.show()

In [ ]:
cols = SPOTLIGHT + TUNING
corr = df[cols].corr()
fig, ax = plt.subplots(figsize=(9,8))
im = ax.imshow(corr, vmin=-1, vmax=1, cmap='RdBu_r')
ax.set_xticks(range(len(cols))); ax.set_xticklabels(cols, rotation=90, fontsize=8)
ax.set_yticks(range(len(cols))); ax.set_yticklabels(cols, fontsize=8)
fig.colorbar(im, label='Pearson r'); ax.set_title('evoked params × tuning correlation')
plt.tight_layout(); plt.show()

## Cluster cells by evoked-response type (optional)

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
X = StandardScaler().fit_transform(df[EVOKED].fillna(df[EVOKED].median()))
km = KMeans(n_clusters=3, n_init=10, random_state=0).fit(X)
df['cluster'] = km.labels_
print(df.groupby('cluster')[['A_sc','A_fast','g_a','R_Squared']].median())
fig, ax = plt.subplots(figsize=(6,4))
for c in sorted(df['cluster'].unique()):
    sub = df[df['cluster']==c]
    ax.scatter(sub['A_fast'], sub['A_sc'], label=f'cluster {c}', s=25)
ax.set_xlabel('A_fast'); ax.set_ylabel('A_sc'); ax.legend(); ax.set_title('response-type clusters')
plt.tight_layout(); plt.show()

## Per-component response decomposition
What each component contributes, via single-component knockouts (`diagnose_mechanistic.decompose`): −SC, −adaptation, −channels, −FC. Shown for the **median-param model** and one **representative cell**.

In [ ]:
import optimization_engine as oe
from diagnose_mechanistic import decompose

S1_KEYS = list(oe.PARAM_NAMES[:10])
# median-param model: median stage-1 (8 structural are constant; I_baseline/I_HD are per-cell) + median evoked
stage1_med = {k: float(df[k].median()) for k in S1_KEYS}
med = [float(df[p].median()) for p in EVOKED]
decompose(stage1_med, med, title='median-param model')

In [ ]:
# one representative cell: best-fit valid cell, overlaid on its OWN PD PSTH
sel = df[df.get('valid', True)] if 'valid' in df else df
row = sel.loc[sel['R_Squared'].idxmax()]
animal, cell = row['Animal_Id'], float(row['Cell_Id'])
ids = list(zip([str(a) for a in npz['animal_ids']], [float(c) for c in npz['cell_ids']]))
i = ids.index((str(animal), cell))
vivo = (npz['bins'], npz['pd_rate'][i], npz['pd_smooth'][i])
stage1_cell = {k: float(row[k]) for k in S1_KEYS}   # per-cell drive + frozen structure
s2 = [float(row[p]) for p in EVOKED]
decompose(stage1_cell, s2, vivo=vivo, title=f'{animal}/{cell}  (R²={row["R_Squared"]:.2f})')

## Mechanistic vs phenomenological, per cell
Join on (Animal_Id, Cell_Id) against the phenomenological `df.csv` to compare fit quality cell-by-cell.

In [ ]:
phen = pd.read_csv('df.csv')[['Animal_Id','Cell_Id','R_Squared']].rename(columns={'R_Squared':'R2_phenom'})
m = df[['Animal_Id','Cell_Id','R_Squared']].rename(columns={'R_Squared':'R2_mech'}).merge(phen, on=['Animal_Id','Cell_Id'])
fig, ax = plt.subplots(figsize=(5,5))
ax.scatter(m['R2_phenom'], m['R2_mech'], s=20, alpha=0.7)
lim = [min(m[['R2_phenom','R2_mech']].min())-0.05, 1.0]
ax.plot(lim, lim, 'k--', alpha=0.4); ax.set_xlim(lim); ax.set_ylim(lim)
ax.set_xlabel('phenomenological R²'); ax.set_ylabel('mechanistic R²'); ax.set_title('per-cell fit quality')
plt.tight_layout(); plt.show()